In [2]:
from numpy import dot,array,pi
import matplotlib.pyplot as plt
import pygame
#all measurements are in meters

pygame.init()


WIDTH, HEIGHT = 1500, 800 #depends on resolution of your computer
SCALE = (WIDTH // 2)/4.5e12 #1/(2.5e10) #calculated with ratio of radius from center of system to Eris and the radius in pixels
CENTER = (WIDTH//2,HEIGHT//2)

display = pygame.display.set_mode((WIDTH, HEIGHT))

def main():
    game = True
    clock = pygame.time.Clock() # regulating frame rate

    DISPLAY_RADIUS = {
    "Earth": 4, "Jupiter": 8, "Saturn": 7,
    "Uranus": 5, "Neptune": 5, "Venus": 4,
    "Mars": 3, "Mercury": 2, "Pluto": 2,
    "Eris": 2, "Haumea": 2, "Makemake": 2,
    "Ceres": 2
    }


    bodies = [
        "Earth", "Jupiter", "Saturn",
        "Uranus", "Neptune", "Venus", 
        "Mars", "Mercury" 
        
    ]
    other = [
        "Eris", "Haumea","Makemake",
        "Ceres", "Pluto"
    ]
    
    radii = {
        "Earth": 6.37e6, "Jupiter": 7.15e7,
        "Saturn": 5.82e7, "Uranus": 2.54e7,
        "Neptune": 2.46e7, "Venus": 6e6,
        "Mars": 3.89e6, "Mercury": 2.44e6,
        "Pluto": 1.19e6, "Eris": 1.163e6,
        "Haumea": 7.8e5, "Makemake": 7.15e5,
        "Ceres": 4.76e5
    }

    colors = {
        "Earth": (52, 235, 73), "Jupiter": (227, 208, 113),
        "Saturn": (117, 110, 74), "Uranus": (204, 252, 242),
        "Neptune": (9, 82, 219), "Venus" : (222, 138, 35),
        "Mars": (232, 75, 23), "Mercury": (163, 158, 158),
        "Pluto": (232, 205, 165), "Eris": (155, 171, 186),
        "Haumea": (182, 187, 191), "Makemake": (163, 80, 94),
        "Ceres": (157, 157, 157)
    }

    trajectories = {}
    t = 0

    
    
    for i in bodies:
        planet = Orbit(i)
        x, y = planet.coords()
        trajectories[i] = list(zip(x,y))
        
    while game:
        clock.tick(60) #60 fps
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                game = False
            
        display.fill((0,0,0))
        pygame.draw.circle(display, (255, 255, 0), CENTER, 8) #sun
        for i in bodies:
            planet = Orbit(i)
            r = planet.distances
            for j in r:
                if i == j:
                    pygame.draw.circle(display, (255,255,255), CENTER, int(r[j]*SCALE), width = 1)
        for i in bodies:
            coordinates = trajectories[i]
            x_t, y_t = coordinates[t % len(coordinates)]

            x_pos = CENTER[0] + x_t * SCALE
            y_pos = CENTER[1] + y_t * SCALE

            pygame.draw.circle(display, colors[i], (x_pos, y_pos), DISPLAY_RADIUS[i]) #DISPLAY_RADIUS[i]
        t += 1
        pygame.display.flip()
        
    pygame.quit()

class Orbit:
    def __init__(self,planet):
        self.planet = planet
        self.distances = { 
            "Jupiter": 7.40595e11, 
            "Saturn": 1.35255e12, 
            "Uranus": 2.74130e12, 
            "Neptune": 4.44445e12, 
            "Earth": 1.47e11, 
            "Venus": 1.07476e11,
            "Mars": 2.06650e11, 
            "Mercury": 4.60012e10, 
            "Pluto": 4.43682e12, 
            "Eris": 5.72274e12, 
            "Haumea" :5.18286e12, 
            "Makemake": 5.69994e12, 
            "Ceres": 3.81678e11
        }

        self.velocities = { 
            "Jupiter": 13720, 
            "Saturn": 10180, 
            "Uranus": 7110, 
            "Neptune": 5500, 
            "Earth": 30290, 
            "Venus": 35260,
            "Mars": 26500, 
            "Mercury": 58980, 
            "Pluto": 6120, 
            "Eris": 5300, 
            "Haumea" :5500, 
            "Makemake": 5200, 
            "Ceres": 19100
        }

        self.bodies = [
            "Earth",
            "Jupiter",
            "Saturn",
            "Uranus",
            "Neptune",
            "Venus",
            "Mars",
            "Mercury",
            "Pluto",
            "Eris",
            "Haumea",
            "Makemake",
            "Ceres"
        ]

        self.semi_major = {
            "Mercury": 5.79092e10,
            "Venus": 1.08209e11,
            "Earth": 1.49598e11,
            "Mars": 2.27944e11,
            "Jupiter": 7.78570e11,
            "Saturn": 1.43353e12,
            "Uranus": 2.87246e12,
            "Neptune": 4.49506e12,
            "Pluto": 5.90638e12,
            "Eris": 1.01618e13,
            "Haumea": 6.484e12,
            "Makemake": 6.850e12,
            "Ceres": 4.137e11
        }
        
    def aGravity(self,r):
        G = 6.67e-11
        M = 1.9891e30
        R = dot(r,r)**0.5 
        a_x = -G*M / R**3 * r[0]
        a_y = -G*M / R**3 * r[1]
        a = array([a_x,a_y], float)

        return a

    def coords(self):
        x = []
        y = []
        r = array([self.distances[self.planet],0], float)
        v = array([0,self.velocities[self.planet]], float)
        G = 6.67e-11
        M = 1.9891e30
        R = dot(r,r)**0.5 
   

        T = ((4*pi**2)*(self.semi_major[self.planet])**3/(G*M))**0.5
        T_earth = ((4*pi**2)*(149.7e9)**3/(G*M))**0.5
        N = 1000 # normalizing 
        dt = T_earth / N
        steps = int(T/dt)
        for i in range(steps):
            x.append(r[0])
            y.append(r[1])
            a = self.aGravity(r)
            r += v * dt + 0.5 * a * dt**2
            a_next = self.aGravity(r)
            v += (( a + a_next )/2) *dt

        return x, y
        #plt.plot(x,y,'c-')
        #plt.axis('square')
        #plt.grid(True)
        #plt.show()
main()

make a note saying that i did not know about nested dicts at the time and that you will hope to include it sometime.